# 15.6 CTR prediction

CTR models turn sparse identities and interactions into a calibrated probability of click.

CTR prediction combines linear terms, explicit crosses, FM-style embedding interactions, and optional tiny deep terms into a logit. Ranking can use the order, but bidding and pacing need calibrated probabilities. Save a copy to Drive to edit.

In [ ]:

import numpy as np
import matplotlib.pyplot as plt

SEED = 1500
rng = np.random.default_rng(SEED)


def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))


def cosine(a, b):
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    denom = np.linalg.norm(a) * np.linalg.norm(b)
    if denom == 0.0:
        return 0.0
    return float(np.dot(a, b) / denom)


def ndcg_at_k(relevance, scores, k=3):
    relevance = np.asarray(relevance, dtype=float)
    scores = np.asarray(scores, dtype=float)
    order = np.argsort(-scores, kind="mergesort")[:k]
    gains = (2.0 ** relevance[order] - 1.0) / np.log2(np.arange(2, len(order) + 2))
    ideal = np.argsort(-relevance, kind="mergesort")[:k]
    ideal_gains = (2.0 ** relevance[ideal] - 1.0) / np.log2(np.arange(2, len(ideal) + 2))
    denom = np.sum(ideal_gains)
    if denom == 0.0:
        return 0.0
    return float(np.sum(gains) / denom)


def rmse(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return float(np.sqrt(np.mean((y_true - y_pred) ** 2)))


def recall_at_k(relevance, scores, k=3):
    relevance = np.asarray(relevance, dtype=float) > 0.0
    scores = np.asarray(scores, dtype=float)
    order = np.argsort(-scores, kind="mergesort")[:k]
    denom = np.sum(relevance)
    if denom == 0:
        return 0.0
    return float(np.sum(relevance[order]) / denom)


def rating_ladder():
    d1 = np.array([
        [5.0, 4.0, 0.0, 1.0],
        [4.0, 5.0, 0.0, 1.0],
        [1.0, 1.0, 5.0, 4.0],
        [0.0, 1.0, 4.0, 5.0],
    ])
    d2 = np.array([
        [5.0, 4.0, 0.0, 1.0, 0.0],
        [4.0, 5.0, 0.0, 1.0, 2.0],
        [1.0, 1.0, 5.0, 4.0, 0.0],
        [0.0, 1.0, 4.0, 5.0, 5.0],
        [5.0, 0.0, 1.0, 0.0, 4.0],
        [0.0, 2.0, 5.0, 4.0, 0.0],
    ])
    d3 = np.array([
        [5.0, 4.0, 0.0, 1.0, 0.0, 2.0],
        [4.0, 4.0, 0.0, 0.0, 2.0, 0.0],
        [1.0, 1.0, 5.0, 4.0, 0.0, 0.0],
        [0.0, 1.0, 4.0, 5.0, 5.0, 0.0],
        [5.0, 0.0, 1.0, 0.0, 4.0, 4.0],
        [0.0, 2.0, 5.0, 4.0, 0.0, 0.0],
        [3.0, 0.0, 3.0, 0.0, 3.0, 0.0],
        [0.0, 4.0, 0.0, 4.0, 0.0, 1.0],
    ])
    d4 = np.array([
        [5.0, 4.0, 0.0, 0.0, 1.0, 0.0, 4.0, 0.0],
        [4.0, 5.0, 0.0, 2.0, 0.0, 0.0, 4.0, 1.0],
        [0.0, 2.0, 5.0, 4.0, 0.0, 4.0, 0.0, 0.0],
        [1.0, 0.0, 4.0, 5.0, 0.0, 5.0, 0.0, 2.0],
        [5.0, 0.0, 1.0, 0.0, 4.0, 0.0, 5.0, 0.0],
        [0.0, 3.0, 5.0, 4.0, 0.0, 4.0, 0.0, 0.0],
        [3.0, 0.0, 0.0, 0.0, 5.0, 1.0, 4.0, 0.0],
        [0.0, 4.0, 0.0, 5.0, 0.0, 5.0, 0.0, 3.0],
        [4.0, 0.0, 2.0, 0.0, 5.0, 0.0, 4.0, 1.0],
        [0.0, 2.0, 4.0, 5.0, 0.0, 4.0, 0.0, 0.0],
    ])
    d5 = np.array([
        [5.0, 4.0, 0.0, 0.0, 1.0, 0.0, 5.0, 0.0, 0.0, 0.0],
        [4.0, 5.0, 0.0, 0.0, 1.0, 0.0, 4.0, 0.0, 0.0, 0.0],
        [5.0, 0.0, 0.0, 0.0, 0.0, 0.0, 5.0, 0.0, 1.0, 0.0],
        [0.0, 0.0, 5.0, 4.0, 0.0, 4.0, 0.0, 0.0, 0.0, 0.0],
        [0.0, 1.0, 4.0, 5.0, 0.0, 5.0, 0.0, 0.0, 0.0, 2.0],
        [0.0, 0.0, 5.0, 0.0, 0.0, 4.0, 0.0, 0.0, 0.0, 0.0],
        [5.0, 0.0, 1.0, 0.0, 4.0, 0.0, 5.0, 0.0, 0.0, 0.0],
        [0.0, 0.0, 0.0, 0.0, 5.0, 0.0, 4.0, 1.0, 0.0, 0.0],
        [0.0, 4.0, 0.0, 4.0, 0.0, 5.0, 0.0, 3.0, 0.0, 0.0],
        [3.0, 0.0, 0.0, 0.0, 5.0, 0.0, 4.0, 0.0, 0.0, 0.0],
        [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 5.0, 4.0],
        [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 4.0, 5.0],
    ])
    return [
        {"name": "D1 tiny rating matrix", "ratings": d1, "holdout": [(0, 2, 5.0)]},
        {"name": "D2 synthetic preferences", "ratings": d2, "holdout": [(0, 2, 5.0), (2, 4, 2.0), (5, 4, 3.0)]},
        {"name": "D3 noise ties sparsity", "ratings": d3, "holdout": [(0, 2, 5.0), (1, 3, 2.0), (6, 1, 3.0), (7, 4, 2.0)]},
        {"name": "D4 inline MovieLens-like sample", "ratings": d4, "holdout": [(0, 2, 4.0), (1, 5, 3.0), (4, 3, 2.0), (6, 7, 2.0)]},
        {"name": "D5 long-tail cold-start", "ratings": d5, "holdout": [(0, 2, 4.0), (2, 9, 2.0), (5, 8, 4.0), (10, 0, 3.0), (11, 1, 3.0)]},
    ]


def slate_ladder():
    return [
        {"name": "D1 3-item slate", "features": np.array([[1.0, 0.0, 0.0], [0.0, 1.0, 1.0], [1.0, 1.0, 0.0]]), "content": np.array([0.744, 0.643, 0.819]), "collab": np.array([0.30, 0.90, 0.70]), "relevance": np.array([2.0, 3.0, 1.0])},
        {"name": "D2 synthetic preferences", "features": np.array([[1.0, 0.1, 0.0], [0.2, 0.9, 0.8], [0.8, 0.7, 0.1], [0.1, 0.2, 1.0]]), "content": np.array([0.70, 0.62, 0.88, 0.55]), "collab": np.array([0.40, 0.85, 0.65, 0.30]), "relevance": np.array([1.0, 3.0, 2.0, 0.0])},
        {"name": "D3 noise ties sparsity", "features": np.array([[1.0, 0.0, 0.2], [0.9, 0.2, 0.0], [0.1, 1.0, 0.9], [0.2, 0.8, 0.8], [0.0, 0.1, 1.0]]), "content": np.array([0.78, 0.78, 0.59, 0.60, 0.51]), "collab": np.array([0.20, 0.50, 0.88, 0.40, 0.10]), "relevance": np.array([2.0, 1.0, 3.0, 2.0, 0.0])},
        {"name": "D4 inline MovieLens-like ratings", "features": np.array([[1.0, 0.3, 0.0], [0.9, 0.4, 0.1], [0.2, 0.9, 0.6], [0.1, 0.6, 1.0], [0.7, 0.2, 0.8], [0.0, 0.1, 0.9]]), "content": np.array([0.82, 0.80, 0.66, 0.59, 0.75, 0.50]), "collab": np.array([0.55, 0.61, 0.92, 0.72, 0.35, 0.25]), "relevance": np.array([2.0, 2.0, 3.0, 1.0, 2.0, 0.0])},
        {"name": "D5 long-tail cold-start", "features": np.array([[1.0, 0.2, 0.0], [0.8, 0.1, 0.1], [0.1, 0.9, 0.8], [0.0, 0.7, 1.0], [0.9, 0.9, 0.0], [0.1, 0.0, 1.0], [1.0, 1.0, 0.0]]), "content": np.array([0.84, 0.78, 0.61, 0.58, 0.90, 0.52, 0.82]), "collab": np.array([0.65, 0.20, 0.91, 0.55, np.nan, 0.05, np.nan]), "relevance": np.array([2.0, 1.0, 3.0, 2.0, 3.0, 0.0, 2.0])},
    ]


def print_ladder_preview(rungs):
    for rung in rungs:
        if "ratings" in rung:
            matrix = rung["ratings"]
            observed = int(np.sum(matrix > 0.0))
            total = int(matrix.size)
            print(rung["name"], matrix.shape, "observed", observed, "of", total)
            print(matrix[:3, :5])
        else:
            relevance = rung["relevance"]
            print(rung["name"], "items", len(relevance), "positives", int(np.sum(relevance > 0.0)))
            print("relevance", relevance[:5])


## The concept, built once: wide, FM, and tiny deep terms

A CTR model outputs $p=\sigma(z)$. A compact Wide & Deep / DeepFM-style logit is $$z=b+w^	op x+\sum_{i\lt j}e_i^	op e_j+f_{deep}(x).$$ The pairwise embedding sum is a real FM interaction, not a placeholder.

In [ ]:

def ctr_score(base=0.0, linear_terms=None, cross_terms=None, embeddings=None, deep=0.0):
    z = float(base)
    if linear_terms is not None:
        z += float(np.sum(linear_terms))
    if cross_terms is not None:
        z += float(np.sum(cross_terms))
    fm_sum = 0.0
    if embeddings is not None:
        embeddings = np.asarray(embeddings, dtype=float)
        for i in range(len(embeddings)):
            for j in range(i + 1, len(embeddings)):
                fm_sum += float(np.dot(embeddings[i], embeddings[j]))
        z += fm_sum
    z += float(deep)
    return {"logit": z, "probability": float(sigmoid(z)), "fm_sum": fm_sum}


def calibrate_logit_for_negative_sampling(sampled_logit, negative_keep_rate):
    return sampled_logit + np.log(negative_keep_rate)


## Check the lesson numbers

The lesson logit $-0.100$ maps to $p=0.475$. A wide cross changes probability from $0.250$ to $0.525$. FM dots sum to $0.670$, and linear $-0.300$ plus FM $0.670$ plus deep $0.250$ gives $z=0.620$ and $p=0.650$.

In [ ]:

base_case = ctr_score(base=-1.2, linear_terms=np.array([0.7, 0.4]))
without_cross = ctr_score(base=-2.0, linear_terms=np.array([0.5, 0.4]))
with_cross = ctr_score(base=-2.0, linear_terms=np.array([0.5, 0.4]), cross_terms=np.array([1.2]))
embeddings = np.array([
    [0.2, 0.5],
    [0.4, 0.1],
    [0.3, 0.6],
])
fm_case = ctr_score(base=-0.3, embeddings=embeddings, deep=0.25)

assert np.round(base_case["probability"], 3) == 0.475
assert np.round(without_cross["probability"], 3) == 0.250
assert np.round(with_cross["probability"], 3) == 0.525
assert np.round(fm_case["fm_sum"], 3) == 0.670
assert np.round(fm_case["logit"], 3) == 0.620
assert np.round(fm_case["probability"], 3) == 0.650

print("base p", round(base_case["probability"], 3))
print("cross p", round(without_cross["probability"], 3), "to", round(with_cross["probability"], 3))
print("DeepFM p", round(fm_case["probability"], 3))


## The dataset ladder

The same slates become CTR examples: content score, collaborative score, and candidate bias feed the logit, while relevance is used to compute NDCG@3 after ranking by predicted CTR.

In [ ]:

rungs = slate_ladder()
print_ladder_preview(rungs)


In [ ]:

rows = []
for rung in rungs:
    collab = np.nan_to_num(rung["collab"], nan=0.25)
    logits = -1.0 + 1.2 * rung["content"] + 0.9 * collab
    probabilities = sigmoid(logits)
    metric = ndcg_at_k(rung["relevance"], probabilities, k=3)
    rows.append({"name": rung["name"], "probabilities": probabilities, "metric": metric, "relevance": rung["relevance"]})

for row in rows:
    print(f"{row['name']}: NDCG@3={row['metric']:.3f}, mean p={np.mean(row['probabilities']):.3f}")


In [ ]:

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
last = rows[-1]
order = np.argsort(-last["probabilities"])
axes[0].bar(np.arange(len(order)), last["probabilities"][order], color="purple")
axes[0].set_title("D5 ranked CTR slate")
axes[0].set_xlabel("rank")
axes[0].set_ylabel("predicted CTR")
axes[1].plot([row["name"].split()[0] for row in rows], [row["metric"] for row in rows], marker="o")
axes[1].set_ylim(0.0, 1.05)
axes[1].set_title("NDCG@3 vs sparsity rung")
axes[1].set_ylabel("NDCG@3")
fig.tight_layout()


## Pitfall on D5: AUC-style ranking with miscalibrated probabilities

A model trained with 10:1 negative downsampling can rank well but overstate CTR unless the intercept is corrected before reading $\sigma(z)$ as probability.

In [ ]:

d5 = slate_ladder()[-1]
collab = np.nan_to_num(d5["collab"], nan=0.25)
sampled_logits = -0.2 + 1.2 * d5["content"] + 0.9 * collab
wrong_prob = sigmoid(sampled_logits)
corrected_logits = calibrate_logit_for_negative_sampling(sampled_logits, negative_keep_rate=0.1)
fixed_prob = sigmoid(corrected_logits)
wrong_metric = ndcg_at_k(d5["relevance"], wrong_prob, k=3)
fixed_metric = ndcg_at_k(d5["relevance"], fixed_prob, k=3)

print("wrong mean CTR", round(float(np.mean(wrong_prob)), 3))
print("corrected mean CTR", round(float(np.mean(fixed_prob)), 3))
print("ranking NDCG@3 before", round(wrong_metric, 3))
print("ranking NDCG@3 after", round(fixed_metric, 3))


## Evaluate it + Practice

Evaluation checklist:
- Track `NDCG@3` on D1--D5 and compare with a no-skill baseline such as popularity or original order.
- Run a sanity check where the strongest observed signal is swapped and the top item changes.
- Ablate the key idea, such as masking, latent factors, calibration, or query-local losses, and confirm the metric drops.
- Watch failure signals: identical recommendations for everyone, head-item dominance, unstable tiny-overlap scores, and poor cold-start behavior.

Practice:
1. Change one D3 tie and predict which slate position moves before running the cell.


2. Add a rare cross term and observe which item jumps.
3. Apply a stronger negative-sampling correction and compare calibration.